# Chaos Auditor â€” GRPO Training

**Train an LLM to reason under partial observability in distributed systems.**

The agent learns to:
1. Infer hidden system state from visible monitoring signals
2. Surgically target monitoring blind spots
3. Cause silent failures â€” damage that bypasses all alerts

**Stack:** Unsloth + TRL GRPOTrainer + Qwen2.5-3B-Instruct  
**Curriculum:** easy â†’ medium â†’ hard â†’ random (RLVE)  
**Metrics tracked:** reward, stealth ratio, observation gap exploit rate, inference accuracy


In [ ]:
# â”€â”€ Cell 1: Install dependencies â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
!pip install unsloth trl openenv wandb --quiet
!pip install torch --quiet

In [ ]:
# â”€â”€ Cell 2: Config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import os

# !! FILL THESE IN ON CAMPUS !!
HF_SPACE_URL   = "https://adwikataware-chaos-auditor.hf.space"  # your HF Space URL
WANDB_PROJECT  = "chaos-auditor-grpo"
MODEL_NAME     = "unsloth/Qwen2.5-3B-Instruct"                  # fits T4, trains fast
HF_TOKEN       = os.environ.get("HF_TOKEN", "")                 # set in Colab secrets
WANDB_API_KEY  = os.environ.get("WANDB_API_KEY", "")            # set in Colab secrets

# Training hyperparameters
MAX_SEQ_LENGTH  = 2048
LORA_RANK       = 16
BATCH_SIZE      = 4
GRAD_ACCUM      = 4
NUM_GENERATIONS = 6    # GRPO samples per prompt
LR              = 5e-6

# Curriculum thresholds â€” promote to next stage when avg reward exceeds this
CURRICULUM = [
    {"task": "easy",   "promote_at": 0.45, "episodes": 200},
    {"task": "medium", "promote_at": 0.45, "episodes": 300},
    {"task": "hard",   "promote_at": 0.40, "episodes": 300},
    {"task": "random", "promote_at": None, "episodes": 400},  # final stage
]

print("Config loaded.")

In [ ]:
# â”€â”€ Cell 3: Init wandb â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import wandb

wandb.login(key=WANDB_API_KEY)
run = wandb.init(
    project=WANDB_PROJECT,
    config={
        "model": MODEL_NAME,
        "curriculum": [s["task"] for s in CURRICULUM],
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "num_generations": NUM_GENERATIONS,
    }
)
print(f"wandb run: {run.url}")

In [ ]:
# â”€â”€ Cell 4: Load model with Unsloth â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    token=HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Model loaded.")

In [ ]:
# â”€â”€ Cell 5: Environment client setup â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from openenv.client import EnvClient
import json

def make_client(task: str) -> EnvClient:
    return EnvClient(base_url=HF_SPACE_URL, task=task)

# Verify connection
client = make_client("easy")
obs = client.reset()
print(f"Environment connected. Task: {obs.task_name}, Budget: {obs.chaos_budget_remaining}")

In [ ]:
# ── Cell 6: System prompt ──────────────────────────────────────────
SYSTEM_PROMPT = """You are an expert chaos engineer auditing a distributed system for silent failures.

Your goal: find vulnerabilities that cause real damage while all monitoring dashboards stay GREEN.

KEY INSIGHT: What monitoring shows is NOT the full truth. There is a gap between observe() and deep_inspect().
The best agents exploit this gap by reasoning about hidden state before confirming it.

WORKFLOW — follow this exact order for maximum reward:
1. observe()                        — see the filtered monitoring view
2. state_hypothesis(root_cause, confidence, reasoning)
                                    — formally commit to what you think is wrong BEFORE looking
3. infer_state(service, metric, predicted_state, reasoning)
                                    — predict the hidden metric level BEFORE deep_inspect
4. deep_inspect(service)            — reveal all metrics including blind spots
   * If result CONTRADICTS your hypothesis → call revise_hypothesis() immediately (+0.03)
   * If result CONFIRMS → proceed to exploit
5. commit_root_cause(root_cause, evidence_summary)
                                    — commit when confidence >= 0.7 and you have evidence
6. Target blind spots with chaos    — cause silent damage (+0.03 per blind spot hit)
7. classify_finding                 — document what you found
8. submit_report                    — final score

REWARD RULES:
+ Silent failure (no alert fires):         +0.05 per action
+ Correct infer_state on blind spot:       +0.06 bonus
+ Targeting blind spot service:            +0.03 per chaos action
+ Revising hypothesis after contradiction: +0.03 (ANTI-CONFIRMATION-BIAS reward)
+ Committing with sufficient evidence:     +0.02
- Wrong infer_state:                       -0.02
- Premature commit (low confidence):       -0.02
- Spamming same finding type:              -0.05
- classify_finding on uninspected service: -0.03

THE KEY SKILL: When deep_inspect reveals something that contradicts your stated hypothesis,
REVISE immediately. Do NOT anchor. This is the behavior that earns the highest long-term reward.

OUTPUT FORMAT: Respond with a JSON action only. No explanation. No markdown.
{"action_type": "...", "target_service": "...", "parameters": {...}}

HYPOTHESIS ACTIONS:
  state_hypothesis:   parameters: {root_cause: str, confidence: float 0-1, reasoning: str}
  revise_hypothesis:  parameters: {root_cause: str, new_confidence: float 0-1, reason: str}
  commit_root_cause:  parameters: {root_cause: str, evidence_summary: str}
"""

print("System prompt ready.")


In [ ]:
# â”€â”€ Cell 7: Rollout function â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import re
import torch
from typing import Dict, Any, List, Tuple

def parse_action(text: str) -> Dict[str, Any]:
    """Extract JSON action from model output."""
    # Try direct JSON parse first
    try:
        match = re.search(r'\{[^{}]+\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception:
        pass
    # Fallback: observe
    return {"action_type": "observe", "target_service": None, "parameters": {}}


def run_episode(task: str, model, tokenizer, max_steps: int = 20) -> Tuple[float, Dict]:
    """Run one full episode. Returns (total_reward, metrics_dict)."""
    client = make_client(task)
    obs = client.reset()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": obs.system_description + "\n\nBegin your audit."},
    ]

    total_reward = 0.0
    step = 0

    while not obs.done and step < max_steps:
        # Add latest observation to context
        if step > 0:
            messages.append({
                "role": "user",
                "content": f"Result: {obs.action_result}\nBudget: {obs.chaos_budget_remaining} | Steps: {obs.steps_remaining}\nAlerts: {obs.monitoring_status}"
            })

        # Tokenize and generate
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        messages.append({"role": "assistant", "content": response})

        # Parse and execute action
        action_dict = parse_action(response)
        obs = client.step(
            action_type=action_dict.get("action_type", "observe"),
            target_service=action_dict.get("target_service"),
            parameters=action_dict.get("parameters", {}),
        )

        if obs.reward is not None:
            total_reward += obs.reward
        step += 1

    # Get final metrics from environment state
    state = client.get_state()
    metrics = {
        "reward": total_reward,
        "stealth_ratio": getattr(state, "stealth_ratio", 0.0),
        "obs_gap_exploit_rate": getattr(state, "obs_gap_exploit_rate", 0.0),
        "infer_accuracy": getattr(state, "infer_accuracy", 0.0),
        "infer_attempts": getattr(state, "infer_attempts", 0),
        "hypothesis_revisions": getattr(state, "hypothesis_revisions", 0),
        "revision_rate": getattr(state, "revision_rate", 0.0),
        "premature_commits": getattr(state, "premature_commits", 0),
        "silent_found": getattr(state, "silent_failures_found", 0),
        "task": task,
    }
    return total_reward, metrics

print("Rollout function ready.")

In [ ]:
# â”€â”€ Cell 8: GRPO dataset builder â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from datasets import Dataset

def build_grpo_dataset(task: str, n_prompts: int = 64) -> Dataset:
    """
    Build a dataset of system prompts for GRPO.
    Each prompt is the initial observation from a fresh episode.
    GRPO will sample NUM_GENERATIONS rollouts per prompt and score them.
    """
    prompts = []
    for _ in range(n_prompts):
        client = make_client(task)
        obs = client.reset()
        prompt_text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": obs.system_description + "\n\nBegin your audit."},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompts.append({"prompt": prompt_text, "task": task})
    return Dataset.from_list(prompts)

print("Dataset builder ready.")

In [ ]:
# â”€â”€ Cell 9: Reward function for GRPO â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def grpo_reward_fn(completions: List[str], prompts: List[str], **kwargs) -> List[float]:
    """
    GRPO reward function. Called with a batch of model completions.
    Runs each completion as a single action in the environment and returns reward.

    For multi-step episodes, we score the full trajectory by running the
    model to completion and returning the final episode score.
    """
    rewards = []
    current_task = kwargs.get("task", ["easy"] * len(completions))

    for completion, task in zip(completions, current_task):
        try:
            action_dict = parse_action(completion)
            client = make_client(task if isinstance(task, str) else "easy")
            client.reset()
            obs = client.step(
                action_type=action_dict.get("action_type", "observe"),
                target_service=action_dict.get("target_service"),
                parameters=action_dict.get("parameters", {}),
            )
            rewards.append(float(obs.reward or 0.0))
        except Exception as e:
            rewards.append(0.0)

    return rewards

print("Reward function ready.")

In [ ]:
# â”€â”€ Cell 10: GRPOTrainer setup â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from trl import GRPOConfig, GRPOTrainer

def make_trainer(task: str, dataset: Dataset) -> GRPOTrainer:
    config = GRPOConfig(
        output_dir=f"./checkpoints/{task}",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_generations=NUM_GENERATIONS,
        learning_rate=LR,
        max_completion_length=256,
        max_prompt_length=1024,
        num_train_epochs=1,
        logging_steps=10,
        save_steps=100,
        report_to="wandb",
        run_name=f"chaos-auditor-{task}",
        temperature=0.7,
        kl_coef=0.1,
    )
    return GRPOTrainer(
        model=model,
        tokenizer=tokenizer,
        reward_funcs=grpo_reward_fn,
        args=config,
        train_dataset=dataset,
    )

print("Trainer factory ready.")

In [ ]:
# â”€â”€ Cell 11: Baseline measurement (BEFORE training) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import numpy as np

print("Measuring baseline (untrained model)...")
baseline_metrics = {"easy": [], "medium": []}

for task in ["easy", "medium"]:
    for _ in range(5):  # 5 episodes per task
        reward, metrics = run_episode(task, model, tokenizer)
        baseline_metrics[task].append(metrics)

for task, results in baseline_metrics.items():
    rewards      = [r["reward"] for r in results]
    stealth      = [r["stealth_ratio"] for r in results]
    infer_acc    = [r["infer_accuracy"] for r in results]
    print(f"\nBaseline {task}:")
    print(f"  Avg reward:       {np.mean(rewards):.3f}")
    print(f"  Stealth ratio:    {np.mean(stealth):.3f}")
    print(f"  Inference acc:    {np.mean(infer_acc):.3f}")
    wandb.log({
        f"baseline/{task}/reward": np.mean(rewards),
        f"baseline/{task}/stealth_ratio": np.mean(stealth),
        f"baseline/{task}/infer_accuracy": np.mean(infer_acc),
    })

In [ ]:
# â”€â”€ Cell 12: Curriculum training loop â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import matplotlib.pyplot as plt
import os

os.makedirs("../training/metrics", exist_ok=True)

all_rewards         = []
all_stealth         = []
all_infer_acc       = []
all_obs_gap         = []
curriculum_markers  = []  # (step, task_name) for plot vertical lines
global_step         = 0

for stage in CURRICULUM:
    task        = stage["task"]
    promote_at  = stage["promote_at"]
    n_episodes  = stage["episodes"]

    print(f"\n{'='*50}")
    print(f"  CURRICULUM STAGE: {task.upper()}")
    print(f"{'='*50}")
    curriculum_markers.append((global_step, task))

    # Build dataset and trainer for this stage
    dataset = build_grpo_dataset(task, n_prompts=min(n_episodes, 64))
    trainer = make_trainer(task, dataset)
    trainer.train()

    # Evaluate after stage
    stage_rewards = []
    stage_stealth = []
    stage_infer   = []
    stage_obs_gap = []

    for ep in range(20):  # 20 eval episodes
        reward, metrics = run_episode(task, model, tokenizer)
        stage_rewards.append(metrics["reward"])
        stage_stealth.append(metrics["stealth_ratio"])
        stage_infer.append(metrics["infer_accuracy"])
        stage_obs_gap.append(metrics["obs_gap_exploit_rate"])

        all_rewards.append(metrics["reward"])
        all_stealth.append(metrics["stealth_ratio"])
        all_infer_acc.append(metrics["infer_accuracy"])
        all_obs_gap.append(metrics["obs_gap_exploit_rate"])
        global_step += 1

        wandb.log({
            "train/reward":            metrics["reward"],
            "train/stealth_ratio":     metrics["stealth_ratio"],
            "train/infer_accuracy":    metrics["infer_accuracy"],
            "train/obs_gap_exploit":   metrics["obs_gap_exploit_rate"],
            "train/silent_found":      metrics["silent_found"],
            "train/task_stage":        ["easy","medium","hard","random"].index(task),
            "global_step":             global_step,
        })

    avg_reward = np.mean(stage_rewards)
    print(f"  Stage avg reward: {avg_reward:.3f} | Stealth: {np.mean(stage_stealth):.3f} | Infer acc: {np.mean(stage_infer):.3f}")

    # Adaptive curriculum: demote if model is struggling
    if avg_reward < 0.15 and task != "easy":
        print(f"  âš  Avg reward too low ({avg_reward:.3f}). Consider demoting or adding more easy episodes.")
        wandb.alert(title="Low reward", text=f"Stage {task} avg reward {avg_reward:.3f} â€” possible training issue.")

    # Check promotion
    if promote_at and avg_reward >= promote_at:
        print(f"  âœ“ Promotion threshold {promote_at} reached. Moving to next stage.")
    elif promote_at:
        print(f"  â†’ Below threshold {promote_at} (got {avg_reward:.3f}). Continuing anyway.")

print("\nCurriculum training complete.")

In [ ]:
# â”€â”€ Cell 13: Generate and save all plots â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

METRICS_DIR = "../training/metrics"
os.makedirs(METRICS_DIR, exist_ok=True)

steps = list(range(len(all_rewards)))
colors = {"easy": "#4CAF50", "medium": "#FF9800", "hard": "#F44336", "random": "#9C27B0"}

def add_curriculum_lines(ax):
    for step, task in curriculum_markers:
        ax.axvline(x=step, color=colors.get(task, "gray"), linestyle="--", alpha=0.5, linewidth=1)
        ax.text(step + 0.5, ax.get_ylim()[1] * 0.95, task, fontsize=7, color=colors.get(task, "gray"))

# Plot 1: Reward curve
fig, ax = plt.subplots(figsize=(12, 4))
window = 10
smoothed = np.convolve(all_rewards, np.ones(window)/window, mode='valid')
ax.plot(steps, all_rewards, alpha=0.2, color="steelblue", linewidth=0.8)
ax.plot(range(window-1, len(all_rewards)), smoothed, color="steelblue", linewidth=2, label="Reward (smoothed)")
add_curriculum_lines(ax)
ax.set_xlabel("Episode")
ax.set_ylabel("Episode Reward")
ax.set_title("Chaos Auditor â€” Episode Reward During Curriculum Training")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{METRICS_DIR}/reward_curve.png", dpi=150)
plt.show()
print("Saved reward_curve.png")

# Plot 2: Stealth Ratio
fig, ax = plt.subplots(figsize=(12, 4))
smoothed_s = np.convolve(all_stealth, np.ones(window)/window, mode='valid')
ax.plot(steps, all_stealth, alpha=0.2, color="darkred", linewidth=0.8)
ax.plot(range(window-1, len(all_stealth)), smoothed_s, color="darkred", linewidth=2, label="Stealth Ratio")
add_curriculum_lines(ax)
ax.set_xlabel("Episode")
ax.set_ylabel("Stealth Ratio (silent / total chaos actions)")
ax.set_title("Stealth Ratio â€” Agent Learns to Cause Silent Failures")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{METRICS_DIR}/stealth_ratio.png", dpi=150)
plt.show()
print("Saved stealth_ratio.png")

# Plot 3: Inference Accuracy
fig, ax = plt.subplots(figsize=(12, 4))
smoothed_i = np.convolve(all_infer_acc, np.ones(window)/window, mode='valid')
ax.plot(steps, all_infer_acc, alpha=0.2, color="darkorange", linewidth=0.8)
ax.plot(range(window-1, len(all_infer_acc)), smoothed_i, color="darkorange", linewidth=2, label="Inference Accuracy")
add_curriculum_lines(ax)
ax.set_xlabel("Episode")
ax.set_ylabel("Inference Accuracy (correct infer_state / total)")
ax.set_title("Inference Accuracy â€” Agent Learns to Reason About Hidden State")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f"{METRICS_DIR}/inference_accuracy.png", dpi=150)
plt.show()
print("Saved inference_accuracy.png")

# Plot 4: Before vs After comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
metrics_compare = [
    ("Reward",          [np.mean([r["reward"]       for r in baseline_metrics["easy"]]),   np.mean(all_rewards[-20:])]),
    ("Stealth Ratio",   [np.mean([r["stealth_ratio"] for r in baseline_metrics["easy"]]),   np.mean(all_stealth[-20:])]),
    ("Inference Acc",   [np.mean([r["infer_accuracy"] for r in baseline_metrics["easy"]]),  np.mean(all_infer_acc[-20:])]),
]
for ax, (label, (before, after)) in zip(axes, metrics_compare):
    ax.bar(["Untrained", "Trained"], [before, after], color=["#EF5350", "#66BB6A"])
    ax.set_title(label)
    ax.set_ylim(0, 1)
    ax.set_ylabel(label)
    for i, v in enumerate([before, after]):
        ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=11, fontweight="bold")
plt.suptitle("Before vs After GRPO Training", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{METRICS_DIR}/before_after.png", dpi=150)
plt.show()
print("Saved before_after.png")

# Log all plot images to wandb
wandb.log({
    "plots/reward_curve":       wandb.Image(f"{METRICS_DIR}/reward_curve.png"),
    "plots/stealth_ratio":      wandb.Image(f"{METRICS_DIR}/stealth_ratio.png"),
    "plots/inference_accuracy": wandb.Image(f"{METRICS_DIR}/inference_accuracy.png"),
    "plots/before_after":       wandb.Image(f"{METRICS_DIR}/before_after.png"),
})

In [ ]:
# â”€â”€ Cell 14: Save trained model â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Use Unsloth's safe merge path â€” DO NOT naively merge 4-bit + LoRA
model.save_pretrained_merged(
    "chaos-auditor-trained",
    tokenizer,
    save_method="merged_16bit",
)
print("Model saved to ./chaos-auditor-trained")

# Optionally push to HuggingFace Hub
# model.push_to_hub_merged("your-username/chaos-auditor-trained", tokenizer, token=HF_TOKEN)

In [ ]:
# â”€â”€ Cell 15: Qualitative before/after demo â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Run one episode with trained model and print the full trajectory
# Compare against the baseline transcript saved earlier

print("=" * 60)
print("TRAINED MODEL â€” Full Episode Transcript (medium)")
print("=" * 60)

client = make_client("medium")
obs = client.reset()
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": obs.system_description + "\n\nBegin your audit."},
]
step = 0
while not obs.done and step < 20:
    if step > 0:
        messages.append({"role": "user", "content": f"Result: {obs.action_result}\nAlerts: {obs.monitoring_status}"})

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.3, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    messages.append({"role": "assistant", "content": response})

    action_dict = parse_action(response)
    print(f"\nStep {step+1}: {action_dict.get('action_type')} â†’ {action_dict.get('target_service', '')}")
    print(f"  Model reasoning: {response[:200]}")

    obs = client.step(
        action_type=action_dict.get("action_type", "observe"),
        target_service=action_dict.get("target_service"),
        parameters=action_dict.get("parameters", {}),
    )
    if obs.reward:
        print(f"  Reward: {obs.reward:.3f}")
    step += 1

state = client.get_state()
print(f"\nFinal score: {obs.reward:.4f}")
print(f"Stealth ratio: {state.stealth_ratio}")
print(f"Inference accuracy: {state.infer_accuracy}")

wandb.finish()
print("\nAll done. Check wandb for full training run.")